<a href="https://colab.research.google.com/github/atenine/LING-230-Single-Model-Disfluency-Detector/blob/multi-encoder-multi-dataset/scripts/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from huggingface_hub import login
from dotenv import dotenv_values
import pandas as pd
from transformers import AutoConfig, AutoModel, WhisperProcessor
import torchaudio
import torch
import torch.nn as nn
import numpy as np
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler

%load_ext tensorboard


login(dotenv_values()['HUGGINGFACE_API_KEY'])

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [3]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

In [4]:
torch.cuda.empty_cache()
import gc
gc.collect()

199

In [5]:
# from google.colab import drive
# drive.mount('../data/drive')

In [6]:
# !pip install gdown

# import os
# import tarfile

# # Ensure ../data/data directory exists for extraction
# os.makedirs('../data/data', exist_ok=True)

# # Google Drive file ID for the zip file provided by the user
# file_id = '18I3VYO_Le-_cFY6Rmbg1MLbHrwmwPE6F'
# zip_file_name = '../data/data/SEP-28k_CLIP.zip'

# print(f"Downloading {zip_file_name} from Google Drive...")
# # Download the file from Google Drive
# !gdown --id "$file_id" -O "$zip_file_name"

# print(f"Extracting {zip_file_name}...")
# # Extract the zip file to ../data/data/
# with tarfile.open(zip_file_name, 'r') as tar:
#     tar.extractall('../data/data/')

# # Define the paths based on the extracted data
# clips_path = '../data/data/SEP-28k_CLIP'


# print("Data setup complete.")
# print(f"Clips path set to: {clips_path}")
# print("Clips directory exists:", os.path.exists(clips_path))

In [7]:
# !pip install gdown

# import os
# import tarfile

# # Ensure ../data/data directory exists for extraction
# os.makedirs('../data/data', exist_ok=True)

# # Google Drive file ID for the zip file provided by the user
# file_id = '15Bb7U28q9IdTRB7vjqjTEzXsDjKBrvsK'
# zip_file_name = '../data/data/FluencyBank.zip'

# print(f"Downloading {zip_file_name} from Google Drive...")
# # Download the file from Google Drive
# !gdown --id "$file_id" -O "$zip_file_name"

# print(f"Extracting {zip_file_name}...")
# # Extract the zip file to ../data/data/
# with tarfile.open(zip_file_name, 'r') as tar:
#     tar.extractall('../data/data/')

# # Define the paths based on the extracted data
# clips_path = '../data/data/fluencybank_CLIP'


# print("Data setup complete.")
# print(f"Clips path set to: {clips_path}")
# print("Clips directory exists:", os.path.exists(clips_path))

In [8]:
# !pip install gdown

# import os
# import tarfile
# import zipfile

# # Ensure ../data/data directory exists for extraction
# os.makedirs('../data/data', exist_ok=True)

# # Google Drive file ID for the zip file provided by the user
# file_id = '1sfmaVCrCNNUwCQm-qWmnZ4cU35lIenmT'
# zip_file_name = '../data/data/UCLASS.zip'

# print(f"Downloading {zip_file_name} from Google Drive...")
# # Download the file from Google Drive
# !gdown --id "$file_id" -O "$zip_file_name"

# print(f"Extracting {zip_file_name}...")
# # Extract the zip file to ../data/data/
# with zipfile.ZipFile(zip_file_name, 'r') as z:
#     z.extractall('../data/data/')

# # Define the paths based on the extracted data
# clips_path = '../data/data/uclass_CLIP'


# print("Data setup complete.")
# print(f"Clips path set to: {clips_path}")
# print("Clips directory exists:", os.path.exists(clips_path))

In [9]:
csv_path   = '../data/SEP-28k-Extended_clips.csv'
clips_path = '../data/data/SEP-28k_CLIP'
print("csv exist:", os.path.exists(csv_path))
print("Clips exist:", os.path.exists(clips_path))

csv exist: True
Clips exist: False


In [10]:
csv_path   = '../data/fluencybank/fluencybank_labels.csv'
clips_path = '../data/data/fluencybank_CLIP'
print("csv exist:", os.path.exists(csv_path))
print("Clips exist:", os.path.exists(clips_path))

csv exist: True
Clips exist: False


In [11]:
csv_path   = '../data/uclass_sep28k-style/uclass_sep28k-style_labels.csv'
clips_path = '../data/uclass_sep28k-style/clips/clips/clips'
print("csv exist:", os.path.exists(csv_path))
print("Clips exist:", os.path.exists(clips_path))

csv exist: True
Clips exist: True


In [12]:
# Check CSV
print("CSV exists:", os.path.exists(csv_path))

# Check one audio clip
path = clips_path + '/HeStutters/10/HeStutters_10_0.wav'
print("Data Exists:", os.path.exists(path))


CSV exists: True
Data Exists: False


In [13]:
import pandas as pd
# fb = pd.read_csv('../data/fluencybank_labels.csv')
# print("Columns:", fb.columns.tolist())
# print(fb.head(2))

In [14]:
# SEP-28k
csv_path_sep28k   = '../data/SEP-28k-Extended_clips.csv'
clips_path_sep28k = '../data/SEP-28k_CLIP'
# csv_path_sep28k   = '/content/SEP-28k-Extended_clips.csv'
# clips_path_sep28k = '/content/data/SEP-28k_CLIP'

df = pd.read_csv(csv_path_sep28k)
df['filepath'] = df.apply(
    lambda row: os.path.join(
        clips_path_sep28k,
        str(row['Show']),
        str(row['EpId']),
        f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ), axis=1
)
df['exists'] = df['filepath'].apply(os.path.exists)
df = df[df['exists']]
print(f"SEP-28k: {len(df)} clips found")

# Label map — defined once
label_map = {
    'Prolongation':    'prolongation',
    'Block':           'block',
    'SoundRep':        'soundrep',
    'WordRep':         'wordrep',
    'Interjection':    'interjection',
    'NoStutteredWords':'fluent'
}
stutter_map = {k: v for k, v in label_map.items() if v != 'fluent'}


SEP-28k: 28177 clips found


In [15]:
# remove rows that do not have at least two annotators agreeing
def create_label(row):
  if any(row[col] >= 2 for col in label_map.keys()):
    return 1
  elif row['NoStutteredWords'] >=2:
    return 0
  else:
    return -1

df['label'] = df.apply(create_label, axis=1)
df = df[df['label'] != -1]

In [16]:
# Normalize SEP-28k labels
df[list(label_map.values())] = df[list(label_map.keys())].div(3.0).rename(columns=label_map)
print("\nAverage probability per stutter type:")
for col in list(stutter_map.values()):
    print(f"  {col}: {df[col].mean():.3f}")

# Train/val/test split — SEP-28k only
subset_fraction = 1
train_df = df[df['SEP28k-T'] == 'train'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
valid_df = df[df['SEP28k-T'] == 'dev'].sample(frac=subset_fraction,   random_state=42).reset_index(drop=True)
test_df  = df[df['SEP28k-T'] == 'test'].sample(frac=subset_fraction,  random_state=42).reset_index(drop=True)

print(f"\nSEP-28k splits:")
print(f"  Train: {len(train_df)}")
print(f"  Val:   {len(valid_df)}")
print(f"  Test:  {len(test_df)}")


Average probability per stutter type:
  prolongation: 0.140
  block: 0.182
  soundrep: 0.101
  wordrep: 0.108
  interjection: 0.235

SEP-28k splits:
  Train: 6205
  Val:   6010
  Test:  14226


In [17]:
# FluencyBank
fb_clips_path = '../data/fluencybank/fluencybank_CLIP/'
fb_df = pd.read_csv('../data/fluencybank/fluencybank_labels.csv')
# fb_clips_path = '/content/data/fluencybank_CLIP/'
# fb_df = pd.read_csv('/content/fluencybank_labels.csv')
fb_df['filepath'] = fb_df.apply(
    lambda row: os.path.join(
        fb_clips_path,
        str(row['Show']),
        str(row['EpId']),
        f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ), axis=1
)
fb_df['exists'] = fb_df['filepath'].apply(os.path.exists)
fb_df = fb_df[fb_df['exists']]
fb_df[list(stutter_map.values())] = fb_df[list(stutter_map.keys())].div(3.0).rename(columns=stutter_map)
print(f"\nFluencyBank: {len(fb_df)} clips found")



FluencyBank: 1859 clips found


In [18]:
# UCLASS
uc_clips_path = '../data/uclass_sep28k-style/clips/clips/clips'
uc_df = pd.read_csv('../data/uclass_sep28k-style/uclass_sep28k-style_labels.csv')
# uc_clips_path = '/content/data/clips/clips'
# uc_df = pd.read_csv('/content/uclass_sep28k-style_labels.csv')
uc_df['filepath'] = uc_df['Clip'].apply(
    lambda clip: os.path.join(uc_clips_path, f"{clip}.wav")
)
uc_df['exists'] = uc_df['filepath'].apply(os.path.exists)
uc_df = uc_df[uc_df['exists']]
uc_df['prolongation'] = uc_df['Prolongation'].astype(float)
uc_df['block']        = uc_df['Block'].astype(float)
uc_df['soundrep']     = uc_df['SoundRep'].astype(float)
uc_df['wordrep']      = uc_df['WordRep'].astype(float)
uc_df['interjection'] = uc_df['Interjection'].astype(float)
print(f"UCLASS: {len(uc_df)} clips found")

UCLASS: 4712 clips found


In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [20]:
# # load voc2vec Encoder
# print("\n Loading voc2vec encoder...")
# # MODEL_NAME = "alkiskoudounas/voc2vec-hubert-ls-pt" # facebook/wav2vec2-base, alkiskoudounas/voc2vec-hubert-ls-pt, openai/whisper-large-v3-turbo
# MODEL_NAME = "facebook/wav2vec2-base"
# # MODEL_NAME = "openai/whisper-large-v3-turbo"
# encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
# encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")

In [21]:
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")

In [22]:
import soundfile as sf
import numpy as np

from transformers import WhisperProcessor, AutoModel
import soundfile as sf
import numpy as np
import shutil
import gc

whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")

# Hyperparameters
batch_size    = 256
num_epochs    = 50
learning_rate = 0.0001

# Paths
SAVE_PATH  = '../data/embeddings/'
LOCAL_PATH = '../data/embeddings/'
os.makedirs(SAVE_PATH,  exist_ok=True)
os.makedirs(LOCAL_PATH, exist_ok=True)

# Encoders
ENCODERS = {
    'wav2vec': 'facebook/wav2vec2-base',
    'voc2vec': 'alkiskoudounas/voc2vec-hubert-ls-pt',
    'whisper': 'openai/whisper-small'
}

def filter_valid(df):
    valid = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Filtering"):
        try:
            if sf.info(row['filepath']).frames == 48000:
                valid.append(row)
        except:
            pass
    return pd.DataFrame(valid).reset_index(drop=True)

# SEP-28k
print("Filtering train...")
train_df_valid = filter_valid(train_df)
print("Filtering val...")
val_df_valid   = filter_valid(valid_df)
print("Filtering test...")
test_df_valid  = filter_valid(test_df)
print(f"Train: {len(train_df_valid)} | Val: {len(val_df_valid)} | Test: {len(test_df_valid)}")

# FluencyBank + UCLASS
print("Filtering FluencyBank...")
fb_df_valid = filter_valid(fb_df)
print("Filtering UCLASS...")
uc_df_valid = filter_valid(uc_df)
print(f"FluencyBank: {len(fb_df_valid)} | UCLASS: {len(uc_df_valid)}")


Filtering train...


Filtering: 100%|██████████| 6205/6205 [00:30<00:00, 206.07it/s]


Filtering val...


Filtering: 100%|██████████| 6010/6010 [00:28<00:00, 210.02it/s]


Filtering test...


...

Train: 6199 | Val: 6002 | Test: 14219
Filtering FluencyBank...


Filtering: 100%|██████████| 1859/1859 [00:08<00:00, 216.09it/s]


Filtering UCLASS...


Filtering: 100%|██████████| 4712/4712 [00:25<00:00, 187.19it/s]

FluencyBank: 1844 | UCLASS: 4711


In [23]:
class StutterDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, is_whisper=False):
        self.df         = dataframe.reset_index(drop=True)
        self.label_cols = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']
        self.labels     = torch.tensor(self.df[self.label_cols].values, dtype=torch.float32)
        self.is_whisper = is_whisper

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sample_rate = torchaudio.load(row['filepath'])
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sample_rate != 16000:
            waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)

        # Handle both SEP-28k/FluencyBank and UCLASS
        if 'Show' in row.index:
            name = f"{row['Show']}_{row['EpId']}_{row['ClipId']}"
        else:
            name = str(row['Clip'])  # UCLASS

        if self.is_whisper:
            inputs = whisper_processor(
                waveform.squeeze().numpy(),
                sampling_rate=16000,
                return_tensors='pt'
            )
            return inputs['input_features'].squeeze(0), self.labels[idx], name
        else:
            return waveform, self.labels[idx], name

def make_raw_loader(df, is_whisper=False):
    return DataLoader(
        StutterDataset(df, is_whisper=is_whisper),
        batch_size=256,
        num_workers=0,
        pin_memory=False,
        shuffle=False
    )

In [24]:
def precompute_embeddings(loader, split_name, encoder, is_whisper=False):
    save_file = SAVE_PATH + f'{split_name}_embeddings.npy'
    if os.path.exists(save_file):
        print(f"✓ {split_name} already exists, skipping")
        return

    encoder.eval()
    names_list  = []
    labels_list = []
    emb_list    = []
    total       = 0
    chunk_idx   = 0
    CHUNK_SIZE  = 50

    indices = torch.tensor([i for i in range(149)]).to(device)

    with torch.no_grad():
        for waveforms, labels, names in tqdm(loader, desc=f"Extracting {split_name}"):
            waveforms = waveforms.to(device)
            with torch.amp.autocast('cuda'):
                if is_whisper:
                    emb = encoder.encoder(waveforms).last_hidden_state
                    emb = torch.index_select(emb, 1, indices) # select only the first 150 items in the "time" dimension
                else:
                    emb = encoder(waveforms.squeeze(1)).last_hidden_state

            emb_list.append(emb.cpu().float().numpy())
            labels_list.append(labels.numpy())
            names_list.extend(names)
            total += len(waveforms)

            if len(emb_list) >= CHUNK_SIZE:
                chunk = np.concatenate(emb_list, axis=0)
                np.save(SAVE_PATH + f'{split_name}_chunk_{chunk_idx}.npy', chunk)
                chunk_idx += 1
                del chunk, emb_list
                emb_list = []
                gc.collect()

            del emb, waveforms
            torch.cuda.empty_cache()

    if emb_list:
        chunk = np.concatenate(emb_list, axis=0)
        np.save(SAVE_PATH + f'{split_name}_chunk_{chunk_idx}.npy', chunk)
        chunk_idx += 1
        del chunk, emb_list
        gc.collect()

    np.save(SAVE_PATH + f'{split_name}_labels.npy', np.concatenate(labels_list))
    np.save(SAVE_PATH + f'{split_name}_names.npy',  np.array(names_list))

    print(f"Saved {chunk_idx} chunks — merging...")
    first_chunk = np.load(SAVE_PATH + f'{split_name}_chunk_0.npy')
    emb_shape   = (total, first_chunk.shape[1], first_chunk.shape[2])
    del first_chunk

    final = np.lib.format.open_memmap(
        SAVE_PATH + f'{split_name}_embeddings.npy',
        mode='w+', dtype=np.float32, shape=emb_shape
    )
    idx = 0
    for i in range(chunk_idx):
        chunk = np.load(SAVE_PATH + f'{split_name}_chunk_{i}.npy')
        final[idx:idx+len(chunk)] = chunk
        idx += len(chunk)
        del chunk
        os.remove(SAVE_PATH + f'{split_name}_chunk_{i}.npy')
        gc.collect()

    del final
    print(f"✓ Saved {split_name}: {emb_shape}")
    gc.collect()

In [25]:
encoder_name = 'wav2vec'
model_path   = 'facebook/wav2vec2-base'
is_whisper   = False

train_loader_raw = make_raw_loader(train_df_valid)
val_loader_raw   = make_raw_loader(val_df_valid)
test_loader_raw  = make_raw_loader(test_df_valid)
fb_loader_raw    = make_raw_loader(fb_df_valid)
uc_loader_raw    = make_raw_loader(uc_df_valid)

encoder = AutoModel.from_pretrained(model_path).to(device)
encoder.eval()

precompute_embeddings(train_loader_raw, f'train_{encoder_name}',            encoder, is_whisper)
precompute_embeddings(val_loader_raw,   f'val_{encoder_name}',              encoder, is_whisper)
precompute_embeddings(test_loader_raw,  f'test_{encoder_name}',             encoder, is_whisper)
precompute_embeddings(fb_loader_raw,    f'test_fluencybank_{encoder_name}', encoder, is_whisper)
precompute_embeddings(uc_loader_raw,    f'test_uclass_{encoder_name}',      encoder, is_whisper)

del encoder
torch.cuda.empty_cache()
gc.collect()
print(f"✓ {encoder_name} done")

Loading weights: 100%|██████████| 211/211 [00:00<?, ?it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\dunna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\module.py:1367: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorCo

Saved 1 chunks — merging...
✓ Saved train_wav2vec: (6199, 149, 768)


Extracting val_wav2vec: 100%|██████████| 24/24 [01:22<00:00,  3.43s/it]


Saved 1 chunks — merging...
✓ Saved val_wav2vec: (6002, 149, 768)


Extracting test_wav2vec: 100%|██████████| 56/56 [03:20<00:00,  3.57s/it]


Saved 2 chunks — merging...
✓ Saved test_wav2vec: (14219, 149, 768)


Extracting test_fluencybank_wav2vec: 100%|██████████| 8/8 [00:25<00:00,  3.15s/it]


Saved 1 chunks — merging...
✓ Saved test_fluencybank_wav2vec: (1844, 149, 768)


Extracting test_uclass_wav2vec: 100%|██████████| 19/19 [01:02<00:00,  3.30s/it]


Saved 1 chunks — merging...
✓ Saved test_uclass_wav2vec: (4711, 149, 768)
✓ wav2vec done


In [26]:
encoder_name = 'voc2vec'
model_path   = 'alkiskoudounas/voc2vec-hubert-ls-pt'
is_whisper   = False

train_loader_raw = make_raw_loader(train_df_valid)
val_loader_raw   = make_raw_loader(val_df_valid)
test_loader_raw  = make_raw_loader(test_df_valid)
fb_loader_raw    = make_raw_loader(fb_df_valid)
uc_loader_raw    = make_raw_loader(uc_df_valid)

encoder = AutoModel.from_pretrained(model_path).to(device)
encoder.eval()

precompute_embeddings(train_loader_raw, f'train_{encoder_name}',            encoder, is_whisper)
precompute_embeddings(val_loader_raw,   f'val_{encoder_name}',              encoder, is_whisper)
precompute_embeddings(test_loader_raw,  f'test_{encoder_name}',             encoder, is_whisper)
precompute_embeddings(fb_loader_raw,    f'test_fluencybank_{encoder_name}', encoder, is_whisper)
precompute_embeddings(uc_loader_raw,    f'test_uclass_{encoder_name}',      encoder, is_whisper)

del encoder
torch.cuda.empty_cache()
gc.collect()
print(f"✓ {encoder_name} done")

Extracting train_voc2vec: 100%|██████████| 25/25 [01:21<00:00,  3.26s/it]


Saved 1 chunks — merging...
✓ Saved train_voc2vec: (6199, 149, 768)


Extracting val_voc2vec: 100%|██████████| 24/24 [01:22<00:00,  3.43s/it]


Saved 1 chunks — merging...
✓ Saved val_voc2vec: (6002, 149, 768)


Extracting test_voc2vec: 100%|██████████| 56/56 [03:23<00:00,  3.64s/it]


Saved 2 chunks — merging...
✓ Saved test_voc2vec: (14219, 149, 768)


Extracting test_fluencybank_voc2vec: 100%|██████████| 8/8 [00:25<00:00,  3.16s/it]


Saved 1 chunks — merging...
✓ Saved test_fluencybank_voc2vec: (1844, 149, 768)


Extracting test_uclass_voc2vec: 100%|██████████| 19/19 [01:00<00:00,  3.21s/it]


Saved 1 chunks — merging...
✓ Saved test_uclass_voc2vec: (4711, 149, 768)
✓ voc2vec done


In [27]:
encoder_name = 'whisper'
model_path   = 'openai/whisper-small'
is_whisper   = True

train_loader_raw = make_raw_loader(train_df_valid, is_whisper=True)
val_loader_raw   = make_raw_loader(val_df_valid,   is_whisper=True)
test_loader_raw  = make_raw_loader(test_df_valid,  is_whisper=True)
fb_loader_raw    = make_raw_loader(fb_df_valid,    is_whisper=True)
uc_loader_raw    = make_raw_loader(uc_df_valid,    is_whisper=True)

encoder = AutoModel.from_pretrained(model_path).to(device)
encoder.eval()

precompute_embeddings(train_loader_raw, f'train_{encoder_name}',            encoder, is_whisper)
precompute_embeddings(val_loader_raw,   f'val_{encoder_name}',              encoder, is_whisper)
precompute_embeddings(test_loader_raw,  f'test_{encoder_name}',             encoder, is_whisper)
precompute_embeddings(fb_loader_raw,    f'test_fluencybank_{encoder_name}', encoder, is_whisper)
precompute_embeddings(uc_loader_raw,    f'test_uclass_{encoder_name}',      encoder, is_whisper)

del encoder
torch.cuda.empty_cache()
gc.collect()
print(f"✓ {encoder_name} done")

Extracting train_whisper: 100%|██████████| 25/25 [12:08<00:00, 29.15s/it]


Saved 1 chunks — merging...
✓ Saved train_whisper: (6199, 149, 768)


Extracting val_whisper: 100%|██████████| 24/24 [11:07<00:00, 27.82s/it]


Saved 1 chunks — merging...
✓ Saved val_whisper: (6002, 149, 768)


Extracting test_whisper: 100%|██████████| 56/56 [27:07<00:00, 29.06s/it]


Saved 2 chunks — merging...
✓ Saved test_whisper: (14219, 149, 768)


Extracting test_fluencybank_whisper: 100%|██████████| 8/8 [03:33<00:00, 26.67s/it]


Saved 1 chunks — merging...
✓ Saved test_fluencybank_whisper: (1844, 149, 768)


Extracting test_uclass_whisper: 100%|██████████| 19/19 [09:03<00:00, 28.62s/it]


Saved 1 chunks — merging...
✓ Saved test_uclass_whisper: (4711, 149, 768)
✓ whisper done


In [28]:
for encoder_name in ENCODERS.keys():
    for split in ['train', 'val', 'test']:
        f = SAVE_PATH + f'{split}_{encoder_name}_embeddings.npy'
        if os.path.exists(f):
            size = os.path.getsize(f) / 1e9
            print(f"{split}_{encoder_name}: {size:.2f} GB")
        else:
            print(f"{split}_{encoder_name}: NOT FOUND")

train_wav2vec: 2.84 GB
val_wav2vec: 2.75 GB
test_wav2vec: 6.51 GB
train_voc2vec: 2.84 GB
val_voc2vec: 2.75 GB
test_voc2vec: 6.51 GB
train_whisper: 2.84 GB
val_whisper: 2.75 GB
test_whisper: 6.51 GB


In [29]:
for encoder_name in ENCODERS.keys():
    for split in ['train', 'val', 'test']:
        for f in ['embeddings', 'labels', 'names']:
            src = SAVE_PATH + f'{split}_{encoder_name}_{f}.npy'
            dst = LOCAL_PATH + f'{split}_{encoder_name}_{f}.npy'

            size = os.path.getsize(src) / 1e9
            if size > 10:  # skip files larger than 10GB
                print(f"Skipping {split}_{encoder_name}_{f}.npy ({size:.1f}GB) — too large")
                continue

            if not os.path.exists(dst):
                print(f"  Copying {split}_{encoder_name}_{f}.npy ({size:.1f}GB)...")
                shutil.copy(src, dst)
            else:
                print(f"  ✓ {split}_{encoder_name}_{f}.npy already local")

print("✓ Done copying")

  ✓ train_wav2vec_embeddings.npy already local
  ✓ train_wav2vec_labels.npy already local
  ✓ train_wav2vec_names.npy already local
  ✓ val_wav2vec_embeddings.npy already local
  ✓ val_wav2vec_labels.npy already local
  ✓ val_wav2vec_names.npy already local
  ✓ test_wav2vec_embeddings.npy already local
  ✓ test_wav2vec_labels.npy already local
  ✓ test_wav2vec_names.npy already local
  ✓ train_voc2vec_embeddings.npy already local
  ✓ train_voc2vec_labels.npy already local
  ✓ train_voc2vec_names.npy already local
  ✓ val_voc2vec_embeddings.npy already local
  ✓ val_voc2vec_labels.npy already local
  ✓ val_voc2vec_names.npy already local
  ✓ test_voc2vec_embeddings.npy already local
  ✓ test_voc2vec_labels.npy already local
  ✓ test_voc2vec_names.npy already local
  ✓ train_whisper_embeddings.npy already local
  ✓ train_whisper_labels.npy already local
  ✓ train_whisper_names.npy already local
  ✓ val_whisper_embeddings.npy already local
  ✓ val_whisper_labels.npy already local
  ✓ val_

In [30]:
import shutil

# Copy from Drive to local Colab disk (much faster reads)
LOCAL_PATH = '../data/embeddings/'
os.makedirs(LOCAL_PATH, exist_ok=True)

print("Copying embeddings to local disk...")
for encoder_name in ENCODERS.keys():
    for split in ['train', 'val', 'test']:
        for f in ['embeddings', 'labels', 'names']:
            src = SAVE_PATH + f'{split}_{encoder_name}_{f}.npy'
            dst = LOCAL_PATH + f'{split}_{encoder_name}_{f}.npy'
            if not os.path.exists(dst):
                print(f"  Copying {split}_{encoder_name}_{f}.npy...")
                shutil.copy(src, dst)
            else:
                print(f"  ✓ {split}_{encoder_name}_{f}.npy already local")
print("✓ All embeddings local")

Copying embeddings to local disk...
  ✓ train_wav2vec_embeddings.npy already local
  ✓ train_wav2vec_labels.npy already local
  ✓ train_wav2vec_names.npy already local
  ✓ val_wav2vec_embeddings.npy already local
  ✓ val_wav2vec_labels.npy already local
  ✓ val_wav2vec_names.npy already local
  ✓ test_wav2vec_embeddings.npy already local
  ✓ test_wav2vec_labels.npy already local
  ✓ test_wav2vec_names.npy already local
  ✓ train_voc2vec_embeddings.npy already local
  ✓ train_voc2vec_labels.npy already local
  ✓ train_voc2vec_names.npy already local
  ✓ val_voc2vec_embeddings.npy already local
  ✓ val_voc2vec_labels.npy already local
  ✓ val_voc2vec_names.npy already local
  ✓ test_voc2vec_embeddings.npy already local
  ✓ test_voc2vec_labels.npy already local
  ✓ test_voc2vec_names.npy already local
  ✓ train_whisper_embeddings.npy already local
  ✓ train_whisper_labels.npy already local
  ✓ train_whisper_names.npy already local
  ✓ val_whisper_embeddings.npy already local
  ✓ val_whisp

In [31]:
# comment only this line for colab
LOCAL_PATH = SAVE_PATH
def load_and_zip_embeddings(set_type, encoders, in_ram=False):
    embedding_set = []

    for encoder in encoders:
        embedding = []

        if in_ram == True:
            embedding = torch.from_numpy(
                    np.load(LOCAL_PATH + set_type + '_' + encoder + '_embeddings.npy')
                ).float()
        else:
            embedding = np.load(LOCAL_PATH + set_type + '_' + encoder + '_embeddings.npy', mmap_mode='r')

        embedding_set.append(embedding)

    if in_ram == True:
        out_embeddings = torch.stack(embedding_set, dim=1)
        # print(out_embeddings.size())
    else:
        out_embeddings = list(zip(*embedding_set))

    return out_embeddings

In [32]:
class EmbeddingDataset(Dataset):
    def __init__(self, split_name, in_ram=False):
        print(f"Loading {split_name}...")

        # potential split_name arguments are the following plus "train/test/val_" at the beginning:
        # 'whisper',
        # 'wav2vec',
        # 'voc2vec',
        # 'whisper_wav2vec',
        # 'whisper_voc2vec',
        # 'wav2vec_voc2vec',
        # 'whisper_wav2vec_voc2vec'
        set_type = split_name.split('_')[0] # gets only the "test/train/val"
        encoders = split_name.split('_')[1:] # excludes the "train/test/val" but gets the encoders


        if in_ram:
            self.embeddings = load_and_zip_embeddings(set_type, encoders, in_ram=True)
            self.labels = torch.from_numpy(
                np.load(LOCAL_PATH + set_type + '_' + encoders[0] + '_labels.npy')
            ).float()
            self.in_ram = True
        else:
            self.embeddings = load_and_zip_embeddings(set_type, encoders, in_ram = False)
            self.labels     = np.load(LOCAL_PATH + set_type + '_' + encoders[0] + '_labels.npy', mmap_mode='r')
            self.in_ram     = False
        self.names = np.load(LOCAL_PATH + set_type + '_' + encoders[0] + '_names.npy', mmap_mode='r')

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        if self.in_ram:
            return self.embeddings[idx], self.labels[idx], str(self.names[idx])
        else:
            return (
                torch.tensor(self.embeddings[idx].copy()),
                torch.tensor(self.labels[idx].copy()),
                str(self.names[idx])
            )

def  make_embedding_loader(split_name, shuffle=False, in_ram=False):
    return DataLoader(
        EmbeddingDataset(split_name, in_ram=in_ram),
        batch_size=batch_size,
        num_workers=0,
        pin_memory=False,
        shuffle=shuffle
    )

# # Usage per encoder:
# # train_loader = make_embedding_loader(f'train_{encoder_name}', shuffle=True, in_ram=True)
# # val_loader   = make_embedding_loader(f'val_{encoder_name}', shuffle=True, in_ram=True)
# # test_loader  = make_embedding_loader(f'test_{encoder_name}', shuffle=True, in_ram=False)

In [33]:
class CrossDatasetEmbeddingDataset(Dataset):
    def __init__(self, split_name, dataset, in_ram=False):
        set_type = split_name.split('_')[0]        # 'test'
        encoders = split_name.split('_')[1:]       # ['wav2vec'] or ['whisper', 'wav2vec'] etc
        prefix   = f'{set_type}_{dataset}'         # 'test_fluencybank' or 'test_uclass'

        print(f"Loading {prefix} for encoders: {encoders}...")

        if in_ram:
            self.embeddings = load_and_zip_embeddings(prefix, encoders, in_ram=True)
            self.labels     = torch.from_numpy(
                np.load(LOCAL_PATH + f'{prefix}_{encoders[0]}_labels.npy')
            ).float()
            self.in_ram = True
        else:
            self.embeddings = load_and_zip_embeddings(prefix, encoders, in_ram=False)
            self.labels     = np.load(LOCAL_PATH + f'{prefix}_{encoders[0]}_labels.npy', mmap_mode='r')
            self.in_ram     = False

        self.names = np.load(LOCAL_PATH + f'{prefix}_{encoders[0]}_names.npy', mmap_mode='r')

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        if self.in_ram:
            return self.embeddings[idx], self.labels[idx], str(self.names[idx])
        else:
            emb = torch.tensor(
                np.concatenate([np.array(e) for e in self.embeddings[idx]], axis=0),
                dtype=torch.float32
            )
            return (
                emb,
                torch.tensor(self.labels[idx].copy(), dtype=torch.float32),
                str(self.names[idx])
            )

def make_cross_dataset_loader(encoder_name, dataset, in_ram=False):
    return DataLoader(
        CrossDatasetEmbeddingDataset(f'test_{encoder_name}', dataset, in_ram=in_ram),
        batch_size=batch_size,
        num_workers=0,
        pin_memory=False,
        shuffle=False
    )

In [34]:
import torch
import torch.nn as nn
import math

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, x):
        scores  = self.attention(x)
        weights = torch.softmax(scores, dim=1)
        pooled  = torch.sum(x * weights, dim=1)
        return pooled


class StutterNet(nn.Module):
    def __init__(self,input_dim=768, num_encoders=1):
        super().__init__()

        self.num_encoders = num_encoders

        # Layer 1 — CNN
        if num_encoders == 1:
            self.cnn = nn.Sequential(
                nn.Conv1d(
                    in_channels=input_dim,
                    out_channels=50,
                    kernel_size=2,
                    padding=0),
                nn.ReLU(),
            )
        else:
            self.cnn = nn.Sequential(
                nn.Conv2d(
                    in_channels=input_dim,
                    out_channels=50,
                    kernel_size=(2, num_encoders),
                    padding=0
                ),
                nn.ReLU(),
            )

        # Layer 2 — Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=50,
            nhead=10,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # Layer 3 — Attention Pooling
        self.attention = AttentionPooling(hidden_size=50)

        # Layer 4 — Dense
        self.dense = nn.Sequential(
            nn.Linear(50, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        # Layer 5 — 5 Output Heads
        self.head_prolongation = nn.Linear(128, 1)
        self.head_block        = nn.Linear(128, 1)
        self.head_soundrep     = nn.Linear(128, 1)
        self.head_wordrep      = nn.Linear(128, 1)
        self.head_interjection = nn.Linear(128, 1)

    def forward(self, x):
        # x: (B, T, 768)

        x = x.squeeze()

        # CNN
        if self.num_encoders == 1:
            x = x.permute(0, 2, 1)          # (B, 768, T)
            x = self.cnn(x)                 # (B, 50, T-1)
            x = x.permute(0, 2, 1)          # (B, T-1, 50)
        else:
            x = x.permute(0, 3, 2, 1)       # (B, 768, T, E)
            x = self.cnn(x)                 # (B, 50, T-1)
            x = x.permute(0, 3, 2, 1)       # (B, 768, T, E)
            x = x.squeeze()

        # Entropy invariance scaling (Liu et al.)
        seq_len = x.size(1)
        scale   = math.log(seq_len) if seq_len > 1 else 1.0
        x = x * scale

        # Transformer
        x = self.transformer(x)          # (B, T-1, 50)

        # Attention Pooling
        x = self.attention(x)            # (B, 50)

        # Dense
        x = self.dense(x)               # (B, 128)

        return {
            'prolongation': self.head_prolongation(x).squeeze(-1),
            'block':        self.head_block(x).squeeze(-1),
            'soundrep':     self.head_soundrep(x).squeeze(-1),
            'wordrep':      self.head_wordrep(x).squeeze(-1),
            'interjection': self.head_interjection(x).squeeze(-1)
        }

model = StutterNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params/1e6:.2f}M")
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Parameters: 0.11M
GPU allocated: 0.01 GB


In [35]:
from torch.utils.tensorboard import SummaryWriter

classes = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']

def train(epoch):
    model.train()
    running_loss = {h: 0 for h in classes}
    correct      = {h: 0 for h in classes}
    total        = {h: 0 for h in classes}

    for data in train_loader:
        inputs  = data[0].to(device, non_blocking=True)
        targets = {h: data[1][:, i].to(device, non_blocking=True)
                   for i, h in enumerate(classes)}

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = 0
            for h in classes:
                head_loss = criterion(outputs[h], targets[h])
                running_loss[h] += head_loss.item()
                loss += head_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        for h in classes:
            predicted = torch.round(torch.sigmoid(outputs[h]))
            total[h]   += targets[h].size(0)
            correct[h] += predicted.eq(targets[h]).sum().item()

    for h in classes:
        writer.add_scalar(f"{h} Loss/train",     running_loss[h] / len(train_loader), epoch)
        writer.add_scalar(f"{h} Accuracy/train", 100. * correct[h] / total[h],        epoch)


def eval(epoch):
    model.eval()
    running_loss = {h: 0 for h in classes}
    correct      = {h: 0 for h in classes}
    total        = {h: 0 for h in classes}

    with torch.no_grad():
        for data in val_loader:
            inputs  = data[0].to(device, non_blocking=True)
            targets = {h: data[1][:, i].to(device, non_blocking=True)
                       for i, h in enumerate(classes)}

            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                for h in classes:
                    head_loss = criterion(outputs[h], targets[h])
                    running_loss[h] += head_loss.item()

            for h in classes:
                predicted = torch.round(torch.sigmoid(outputs[h]))
                total[h]   += targets[h].size(0)
                correct[h] += predicted.eq(targets[h]).sum().item()

    avg_val_loss = sum(running_loss.values()) / (len(val_loader) * len(classes))

    for h in classes:
        writer.add_scalar(f"{h} Loss/val",     running_loss[h] / len(val_loader), epoch)
        writer.add_scalar(f"{h} Accuracy/val", 100. * correct[h] / total[h],      epoch)

    return avg_val_loss


In [36]:
training_results = {}

encoder_configs = [
    'wav2vec',
    'voc2vec',
    'whisper',
    'whisper_wav2vec',
    'whisper_voc2vec',
    'wav2vec_voc2vec',
    'whisper_wav2vec_voc2vec'
]

# Create the models directory if it doesn't exist
import os
os.makedirs('../models', exist_ok=True)

for encoder_name in encoder_configs:
    print(f"\n{'='*50}")
    print(f"Training with: {encoder_name}")
    print(f"{'='*50}")

    train_loader = make_embedding_loader(f'train_{encoder_name}', shuffle=True, in_ram=True)
    val_loader   = make_embedding_loader(f'val_{encoder_name}',   shuffle=False, in_ram=True)

    num_encoders = len(encoder_name.split('_'))
    model     = StutterNet(input_dim=768, num_encoders=num_encoders).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    scaler    = GradScaler(device=device)
    writer    = SummaryWriter(log_dir='../runs/', comment=f"_{encoder_name}")

    best_val_loss         = float('inf')
    epochs_no_improvement = 0

    for epoch in tqdm(range(1, num_epochs + 1), desc=f"Epoch [{encoder_name}]"):
        train(epoch)
        eval_loss = eval(epoch)

        if eval_loss < best_val_loss:
            best_val_loss         = eval_loss
            epochs_no_improvement = 0
            torch.save({
                'epoch':        epoch,
                'model':        model.state_dict(),
                'optimizer':    optimizer.state_dict(),
                'val_loss':     best_val_loss,
                'encoder':      encoder_name,
                'input_dim':    768,
                'num_encoders': num_encoders
            }, f'../models/best_{encoder_name}.pth')
        else:
            epochs_no_improvement += 1

        if epochs_no_improvement > 10:
            print("Early stopping...")
            break

        torch.cuda.empty_cache()

    writer.close()
    training_results[encoder_name] = best_val_loss
    print(f"✓ {encoder_name} done — val_loss: {best_val_loss:.4f}")

# Compare
print("\n" + "="*50)
print("TRAINING RESULTS:")
print("="*50)
for name, val_loss in training_results.items():
    print(f"{name:30} - val_loss: {val_loss:.4f}")


Training with: wav2vec
Loading train_wav2vec...
Loading val_wav2vec...


Epoch [wav2vec]: 100%|██████████| 50/50 [01:38<00:00,  1.97s/it]


✓ wav2vec done — val_loss: 0.3681

Training with: voc2vec
Loading train_voc2vec...
Loading val_voc2vec...


Epoch [voc2vec]:  88%|████████▊ | 44/50 [01:28<00:12,  2.02s/it]

Early stopping...
✓ voc2vec done — val_loss: 0.3760

Training with: whisper
Loading train_whisper...


...

Epoch [wav2vec_voc2vec]:  74%|███████▍  | 37/50 [01:57<00:41,  3.18s/it]

Early stopping...
✓ wav2vec_voc2vec done — val_loss: 0.3657

Training with: whisper_wav2vec_voc2vec
Loading train_whisper_wav2vec_voc2vec...


Loading val_whisper_wav2vec_voc2vec...


Epoch [whisper_wav2vec_voc2vec]:  60%|██████    | 30/50 [02:13<01:28,  4.44s/it]

Early stopping...
✓ whisper_wav2vec_voc2vec done — val_loss: 0.2849

TRAINING RESULTS:
wav2vec                        - val_loss: 0.3681
voc2vec                        - val_loss: 0.3760
whisper                        - val_loss: 0.2857
whisper_wav2vec                - val_loss: 0.2883
whisper_voc2vec                - val_loss: 0.2873
wav2vec_voc2vec                - val_loss: 0.3657
whisper_wav2vec_voc2vec        - val_loss: 0.2849


In [37]:
#  SEP-28k test + cross-dataset test
class_to_name = {
    'prolongation': 'Prolongation',
    'block':        'Block',
    'soundrep':     'SoundRep',
    'wordrep':      'WordRep',
    'interjection': 'Interjection'
}

def run_evaluation(model, test_loader, encoder_name, dataset_name):
    total             = {h: 0 for h in classes}
    correct           = {h: 0 for h in classes}
    running_loss      = {h: 0 for h in classes}
    predicted_stutter = {h: 0 for h in classes}
    labels_stutter    = {h: 0 for h in classes}
    correct_stutter   = {h: 0 for h in classes}
    criterion = nn.BCEWithLogitsLoss()
    records = {
        'Clip': [],
        'Prolongation': [],
        'Block': [],
        'SoundRep': [],
        'WordRep': [],
        'Interjection': []
    }

    with torch.no_grad():
        for data in tqdm(test_loader, desc=f"Testing {dataset_name}"):
            inputs  = data[0].to(device, non_blocking=True)
            targets = {h: data[1][:, i].to(device, non_blocking=True)
                       for i, h in enumerate(classes)}

            with torch.amp.autocast('cuda'):
                outputs = model(inputs)

            records['Clip'].extend(data[2])

            for h in classes:
                head_loss        = criterion(outputs[h], targets[h])
                running_loss[h] += head_loss.item()
                probs            = torch.sigmoid(outputs[h])
                predicted_labels = torch.round(probs)
                total[h]        += targets[h].size(0)
                correct[h]      += predicted_labels.eq(targets[h]).sum().item()
                predicted_stutter[h] += predicted_labels.sum().item()
                labels_stutter[h]    += targets[h].sum().item()
                correct_stutter[h]   += (predicted_labels * targets[h]).sum().item()
                records[class_to_name[h]].extend(probs.tolist())

    print(f"\n{encoder_name} → {dataset_name} Results:")
    print("="*40)
    for h in classes:
        accu      = 100. * correct[h] / total[h]
        precision = correct_stutter[h] / predicted_stutter[h] if predicted_stutter[h] > 0 else 0
        recall    = correct_stutter[h] / labels_stutter[h]    if labels_stutter[h]    > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        print(f"  {h:15} F1: {f1:.4f} | P: {precision:.4f} | R: {recall:.4f} | Acc: {accu:.2f}%")

    results_df = pd.DataFrame.from_dict(records)

    # save to both local and Drive
    results_df.to_csv(f'../results/{encoder_name}_{dataset_name}.csv')
    results_df.to_csv(f'/content/drive/MyDrive/stutter_embeddings/results_{encoder_name}_{dataset_name}.csv')
    print(f"✓ Saved results_{encoder_name}_{dataset_name}.csv")

In [38]:
# Load best model for each encoder and test
for encoder_name in encoder_configs:
    print(f"\n{'='*50}")
    print(f"Testing: {encoder_name}")
    print(f"{'='*50}")

    checkpoint = torch.load(f'../models/best_{encoder_name}.pth')

    print(checkpoint['num_encoders'])
    model = StutterNet(
        input_dim=checkpoint['input_dim'],
        num_encoders=checkpoint['num_encoders']
    ).to(device)
    model.load_state_dict(checkpoint['model'])
    model.eval()

    # SEP-28k test
    test_loader = make_embedding_loader(f'test_{encoder_name}', in_ram=True)
    run_evaluation(model, test_loader, encoder_name, 'sep28k')

    # FluencyBank test
    fb_loader = make_cross_dataset_loader(encoder_name, 'fluencybank', in_ram=True)
    run_evaluation(model, fb_loader, encoder_name, 'fluencybank')

    # UCLASS test
    uc_loader = make_cross_dataset_loader(encoder_name, 'uclass', in_ram=True)
    run_evaluation(model, uc_loader, encoder_name, 'uclass')


Testing: wav2vec
1
Loading test_wav2vec...


Testing sep28k: 100%|██████████| 56/56 [00:02<00:00, 26.12it/s]



wav2vec → sep28k Results:
  prolongation    F1: 0.1855 | P: 0.6024 | R: 0.1096 | Acc: 71.72%
  block           F1: 0.0239 | P: 0.3827 | R: 0.0123 | Acc: 61.20%
  soundrep        F1: 0.1528 | P: 0.4904 | R: 0.0905 | Acc: 81.69%
  wordrep         F1: 0.0393 | P: 0.4254 | R: 0.0206 | Acc: 83.55%
  interjection    F1: 0.5225 | P: 0.7508 | R: 0.4006 | Acc: 68.96%
✓ Saved results_wav2vec_sep28k.csv
Loading test_fluencybank for encoders: ['wav2vec']...


Testing fluencybank: 100%|██████████| 8/8 [00:00<00:00, 29.39it/s]



wav2vec → fluencybank Results:
  prolongation    F1: 0.1602 | P: 0.6222 | R: 0.0920 | Acc: 77.87%
  block           F1: 0.0021 | P: 0.1111 | R: 0.0011 | Acc: 62.36%
  soundrep        F1: 0.1375 | P: 0.5299 | R: 0.0790 | Acc: 73.48%
  wordrep         F1: 0.0212 | P: 0.5333 | R: 0.0108 | Acc: 76.84%
  interjection    F1: 0.4427 | P: 0.7345 | R: 0.3169 | Acc: 64.10%
✓ Saved results_wav2vec_fluencybank.csv
Loading test_uclass for encoders: ['wav2vec']...


...


whisper_wav2vec_voc2vec → sep28k Results:
  prolongation    F1: 0.3604 | P: 0.6202 | R: 0.2540 | Acc: 72.33%
  block           F1: 0.1405 | P: 0.5244 | R: 0.0811 | Acc: 61.32%
  soundrep        F1: 0.4620 | P: 0.6725 | R: 0.3519 | Acc: 83.21%
  wordrep         F1: 0.6887 | P: 0.7481 | R: 0.6380 | Acc: 87.67%
  interjection    F1: 0.7612 | P: 0.7980 | R: 0.7276 | Acc: 75.01%
✓ Saved results_whisper_wav2vec_voc2vec_sep28k.csv
Loading test_fluencybank for encoders: ['whisper', 'wav2vec', 'voc2vec']...


Testing fluencybank: 100%|██████████| 8/8 [00:00<00:00,  8.81it/s]



whisper_wav2vec_voc2vec → fluencybank Results:
  prolongation    F1: 0.3356 | P: 0.5686 | R: 0.2381 | Acc: 78.04%
  block           F1: 0.0883 | P: 0.5357 | R: 0.0481 | Acc: 62.58%
  soundrep        F1: 0.4736 | P: 0.6366 | R: 0.3771 | Acc: 75.11%
  wordrep         F1: 0.5805 | P: 0.7054 | R: 0.4932 | Acc: 80.64%
  interjection    F1: 0.7340 | P: 0.7713 | R: 0.7001 | Acc: 70.66%
✓ Saved results_whisper_wav2vec_voc2vec_fluencybank.csv
Loading test_uclass for encoders: ['whisper', 'wav2vec', 'voc2vec']...


Testing uclass: 100%|██████████| 19/19 [00:01<00:00,  9.84it/s]



whisper_wav2vec_voc2vec → uclass Results:
  prolongation    F1: 0.1020 | P: 0.2069 | R: 0.0677 | Acc: 88.79%
  block           F1: 0.1310 | P: 0.6959 | R: 0.0723 | Acc: 70.98%
  soundrep        F1: 0.2761 | P: 0.6961 | R: 0.1722 | Acc: 68.39%
  wordrep         F1: 0.3929 | P: 0.3362 | R: 0.4727 | Acc: 89.77%
  interjection    F1: 0.6409 | P: 0.7320 | R: 0.5699 | Acc: 87.39%
✓ Saved results_whisper_wav2vec_voc2vec_uclass.csv
